### Overview 

This notebook is intended to:

* Analyze the difference between the contracted  capacity and the actual load on a grid section downstream from a feeder. 

This use case is designed as follows:

* Define a grid section by choosing a feeder of interest.
* Display all meters in the grid with their parent transformers and relevant attributes.
* Calculate the contracted capacity by summing the meters' Master Circuit Breaker (MCB) values.
* Use the meters' hourly net consumer load time series to retrieve and aggregate the daily max of hourly-average power in this grid section.
* Display a time series of the aggregated daily maximum net consumer load and the contracted capacity.

Utility companies can use the results of such analysis to determine if it is possible to connect more loads in this section of the grid, analyze the distribution of resources, and increase grid efficiency.

For more details about master circuit breaker analysis and how it can be analyzed using Awesense's platform, please refer to the [UC17 - Actual vs. Contracted Grid Capacity Analysis](https://github.com/Awesense/edm-app-examples/blob/master/use_cases/usecase_descriptions/UC17%20-%20Actual%20vs.%20Contracted%20Grid%20Capacity%20Analysis.pdf) document.

---

### Setup

In [1]:
import getpass
import math
import pandas as pd
import os
from dotenv import load_dotenv
import psycopg2
import plotly.express as px

pd.set_option("display.max_columns", None)

#### Connection 

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials or have any trouble connecting, please contact api@awesense.com. <span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
# Checks for Google Colab: If detected,
# it bypasses reading credentials from local files and securely draws from Colab Secrets.
try:
    from google.colab import userdata
    from google.colab.userdata import SecretNotFoundError
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from contextlib import suppress
    print('☁️ Running in Google Colab. Using Colab Secrets for SQLAPI connection')
    # Look for Colab Secrets, fall back to interactive prompts if missing
    with suppress(SecretNotFoundError): os.environ['EDM_HOST'] = userdata.get('EDM_HOST')
    with suppress(SecretNotFoundError): os.environ['EDM_USER'] = userdata.get('EDM_USER')
    with suppress(SecretNotFoundError): os.environ['EDM_PASSWORD'] = userdata.get('EDM_PASSWORD')
else:
    # If running locally, loads connection parameters from a `.env` file (in this directory or parent directories)
    load_dotenv()

# Prompt the user to manually enter any missing connection parameters not found in `.env` file / Google Colab Secrets
if 'EDM_HOST' not in os.environ: os.environ['EDM_HOST'] = input('EDM server address: ').strip()
if 'EDM_USER' not in os.environ: os.environ['EDM_USER'] = input('EDM username: ')

# psycopg2/libpq automatically read the PG* environment variables, so map the shared EDM_* names onto them.
# (The password is only mapped if provided via `.env`/Colab; otherwise libpq falls back to `~/.pgpass`.)
os.environ['PGHOST'] = os.environ['EDM_HOST']
os.environ['PGUSER'] = os.environ['EDM_USER']
os.environ.setdefault('PGDATABASE', 'edm')
if 'EDM_PASSWORD' in os.environ: os.environ['PGPASSWORD'] = os.environ['EDM_PASSWORD']

print('Verifying SQLAPI connection parameters/credentials with connection attempt')

try:
    # Test database connection
    conn = psycopg2.connect('')
    conn.close()
    print('✅ SQLAPI connection parameters/credentials verified')
except psycopg2.OperationalError:
    print('⚠️ SQLAPI credentials check failed (assume missing/incorrect password, but double-check `.env` / secrets!)')
    # Prompt user securely for password
    os.environ['PGPASSWORD'] = getpass.getpass('Enter EDM Password manually: ')

# Keeps `%sql` result tables rendering correctly on newer versions of prettytable
import prettytable
if 'DEFAULT' not in vars(prettytable): prettytable.DEFAULT = prettytable.TableStyle.DEFAULT

# Load the SQL extension and connect
get_ipython().run_line_magic('reload_ext', 'sql')
get_ipython().run_line_magic('sql', 'postgresql://')

Verifying SQLAPI connection parameters/credentials with connection attempt


✅ SQLAPI connection parameters/credentials verified


### Use Case - Actual vs. Contracted Grid Capacity Analysis 

#### Input Parameters 

Input the grid name.

In [3]:
# PARAMETERS (Cell Tag: parameters)
# Do not split this cell structure. Papermill injects automated
# runtime overrides immediately below this block.
# If you don't want to be asked for input for these parameters, you can set them here.
grid_id = None  # Target grid identifier (e.g., 'awefice')
top_feeder_transformer = None  # Target feeder ID (e.g., 'transformer_2')

In [4]:
# Short-circuit check: If Papermill didn't inject values, prompt the user.
# To skip these prompts, set the parameter values directly in the parameters cell above (PAPERMILL PARAMETERS).
grid_id = grid_id or input("Enter grid ID: ").strip()  # awefice

# Define the grid_id time zone by matching on whether the grid_id contains a known substring.
time_zones = {
    "awefice": "America/Vancouver",
    "SAF": "America/Denver",
    "GSO": "America/New_York",
}
time_zone = next((tz for name, tz in time_zones.items() if name in grid_id), None)
if time_zone is None:
    raise ValueError(f"No time zone mapping found for grid_id '{grid_id}'.")


Fetch and display all top feeders (transformers) in this grid.

In [5]:
%%sql result_feeders <<

SELECT grid_element_id as top_feeder_transformer
FROM grid_element
WHERE grid_id = :grid_id
    AND type = 'Transformer'
    AND is_producer = true;

 * postgresql://


2 rows affected.
Returning data to local variable result_feeders


In [6]:
# Convert the results to a data frame.
df_feeders = result_feeders.DataFrame()

# Display the results.
df_feeders

,top_feeder_transformer
0,transformer_2
1,transformer_6


Input top feeder name.

In [7]:
# Short-circuit check: If Papermill didn't inject values, prompt the user.
# To skip these prompts, set the parameter values directly in the parameters cell above (PAPERMILL PARAMETERS).
top_feeder_transformer = (
    top_feeder_transformer or input("Enter feeder ID: ").strip()
)  # transformer_2

### Data 

#### Meters' Information

Fetch all meters in the grid and their relevant information.

In [8]:
%%sql result_meters <<

SELECT ge_meter.grid_element_id as meter_id,
    ge_meter.phases,
    ge_meter.meta ->> 'maximal_demand' as maximal_demand,
    ge_meter.meta ->> 'parent_transformer_id' as parent_transformer,
    ge_trans.meta ->> 'secondary_voltage' as secondary_voltage
FROM grid_element ge_trans
JOIN grid_element ge_meter
    ON ge_trans.grid_element_id = ge_meter.meta ->> 'parent_transformer_id'
JOIN grid_get_downstream(:grid_id, :top_feeder_transformer, 'false') ggd
    ON ggd.grid_element_id = ge_trans.grid_element_id
WHERE ge_meter.type = 'Meter'
ORDER BY LENGTH(ge_meter.phases), ge_meter.phases,
        cast(substring(ge_meter.grid_element_id, 3, 2) as int) asc;

 * postgresql://


15 rows affected.
Returning data to local variable result_meters


In [9]:
# Convert the results to a data frame.
df_meters = result_meters.DataFrame()

# Change the variables' type.
df_meters = df_meters.astype(
    {"phases": "str", "maximal_demand": "int", "secondary_voltage": "int"}
)

# Calculate the master circuit breaker values based on the meters' phase.
df_meters.loc[df_meters["phases"].isin(["A", "B", "C"]), "master_circuit_kW"] = (
    (df_meters["secondary_voltage"] / math.sqrt(3))
    * df_meters["maximal_demand"]
    * 0.98
    / 1000
)

df_meters.loc[df_meters["phases"].isin(["ABC"]), "master_circuit_kW"] = (
    math.sqrt(3)
    * df_meters["secondary_voltage"]
    * df_meters["maximal_demand"]
    * 0.98
    / 1000
)

# Set up a multi-index data frame to display the meter's phases, meter IDs,
# information necessary to calculate MCB values, and the MCB values.
df_meters = df_meters.set_index(["phases", "meter_id"])

# Display the results.
df_meters

maximal_demand parent_transformer  secondary_voltage  \
phases meter_id                                                         
A      m_1                   30     transformer_16                240   
       m_2                   30     transformer_16                240   
       m_3                   30     transformer_16                240   
B      m_13                  30     transformer_31                120   
       m_14                  30     transformer_31                120   
       m_15                  30     transformer_31                120   
C      m_4                   30     transformer_26                240   
       m_5                   30     transformer_26                240   
       m_6                   30     transformer_26                240   
ABC    m_7                   30     transformer_36                240   
       m_8                   30     transformer_36                240   
       m_9                   30     transformer_36                240   
       m_10                  50     transformer_36                240   
       m_11                  30     transformer_43                480   
       m_12                 150     transformer_47                480   

                 master_circuit_kW  
phases meter_id                     
A      m_1                4.073783  
       m_2                4.073783  
       m_3                4.073783  
B      m_13               2.036892  
       m_14               2.036892  
       m_15               2.036892  
C      m_4                4.073783  
       m_5                4.073783  
       m_6                4.073783  
ABC    m_7               12.221350  
       m_8               12.221350  
       m_9               12.221350  
       m_10              20.368917  
       m_11              24.442701  
       m_12             122.213505

In [10]:
print(
    "The sum of contracted capacity in this section of the grid is {} kW".format(
        round(df_meters["master_circuit_kW"].sum(), 2)
    )
)

The sum of contracted capacity in this section of the grid is 234.24 kW


#### Meters' Hourly Time Series 

Fetch all meters downstream from the top feeder and their hourly net consumer load time series.

In [11]:
%%sql result_system <<

SELECT tdss.timestamp at time zone :time_zone as timestamp,
        ggd.grid_element_id as meter_id,
        tdss.value as "kWh",
        geds.type
FROM grid_get_downstream(:grid_id, :top_feeder_transformer) AS ggd
LEFT JOIN grid_element_data_source geds
    ON geds.grid_id = ggd.grid_id
    AND geds.grid_element_id = ggd.grid_element_id
JOIN ts_data_source_select(geds.grid_element_data_source_id, 'kWh') tdss
    ON TRUE
WHERE ggd.type = 'Meter'
    AND geds.type = 'CONSUMER'
ORDER BY tdss.timestamp, meter_id;

 * postgresql://


788696 rows affected.


Returning data to local variable result_system


In [12]:
# Convert the SQL result to a Python dataframe.
df_system = result_system.DataFrame()

# Aggregate loads per meter and sum the hourly net consumer loads, and rename the kWh column.
df_system_agg = (
    df_system.groupby(["timestamp"])["kWh"]
    .sum()
    .reset_index()
    .rename(columns={"kWh": "sum_kWh"})
)

# Aggregate the sum of the hourly load by days and find the maximum load.
df_system_agg_daily = (
    df_system_agg.groupby(pd.Grouper(key="timestamp", freq="D"))
    .max()
    .reset_index()
)

In [13]:
# plot the aggregated load and the sum of the MCB values.
fig = px.line(
    df_system_agg_daily,
    x="timestamp",
    y="sum_kWh",
    title="Daily Max of Hourly-Average Power in the {} grid Downstream of Top Feeder {} ".format(
        grid_id, top_feeder_transformer
    ),
    labels={
        "timestamp": "Timestamp (Days)",
        "sum_kWh": "Daily Max of Hourly-Average Power (kW)",
    },
)

# Add annotation to indicate the maximum point.
fig.add_annotation(
    x=df_system_agg_daily["timestamp"][df_system_agg_daily["sum_kWh"].idxmax()],
    y=df_system_agg_daily["sum_kWh"].max(),
    text="Max Point = {} kW occurred at {} <br> The ratio of Max Point to sum of contracted capacity is {}".format(
        round(df_system_agg_daily["sum_kWh"].max(), 2),
        df_system_agg_daily["timestamp"][
            df_system_agg_daily["sum_kWh"].idxmax()
        ].strftime("%Y-%m-%d"),
        round(
            (
                df_system_agg_daily["sum_kWh"].max()
                / df_meters["master_circuit_kW"].sum()
            ),
            2,
        ),
    ),
    arrowhead=2,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor="#636363",
    ay=-60,
    font=dict(family="sans serif", size=18, color="black"),
)

# Add a horizontal line showing the sum of contracted capacity.
fig.add_hline(
    y=df_meters["master_circuit_kW"].sum(),
    annotation_text="Sum of Contracted Capacity = {} kW".format(
        round(df_meters["master_circuit_kW"].sum(), 2)
    ),
    annotation_position="top right",
    line_color="Black",
    annotation_font_color="Black",
)
fig.show()

The plot above shows the daily max of hourly-average power of all meters downstream from the feeder. The horizontal line shows the sum of the contracted capacity in the grid section, and the arrow points out the maximum point on the graph. The difference between the power time series and the horizontal line represents the average available capacity. Utility companies can use this to determine if more load can be added to this grid section or to reduce the contractual load and use the resources elsewhere. 

---